In [4]:
import cv2
import numpy as np
import pandas as pd
import math
import os
import re

def classify_function(image_name):
    """Classify function type based on image name."""
    trig_functions = {'sin', 'cos', 'tan', 'cot', 'sec', 'csc'}
    polynomial_functions = {'linear', 'quadratic', 'cubic', 'quartic', 'quintic'}
    exponential_functions = {'exponential', 'exp_2x', 'exp_neg_x'}
    logarithmic_functions = {'log', 'log10'}
    hyperbolic_functions = {'sinh', 'cosh', 'tanh'}
    rational_functions = {'rational_1', 'rational_2'}
    piecewise_functions = {'piecewise'}

    # Extract base name without numbers or file extensions
    base_name = re.sub(r'\d+', '', image_name).lower()

    if base_name in trig_functions:
        return f'Trigonometric'
    elif base_name in polynomial_functions:
        return f'Polynomial'
    elif base_name in exponential_functions:
        return 'Exponential'
    elif base_name in logarithmic_functions:
        return 'Logarithmic'
    elif base_name in hyperbolic_functions:
        return 'Hyperbolic'
    elif base_name in rational_functions:
        return 'Rational'
    elif base_name in piecewise_functions:
        return 'Piecewise'
    else:
        return 'Unknown'

def extract_graph_data(image_path, graph_x_range=(-5, 5), graph_y_range=(-10, 10)):
    """Extracts data points from a graph image and returns a DataFrame."""
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"Could not load {image_path}. Skipping...")
        return None

    # Apply adaptive thresholding
    thresh = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)

    # Detect edges
    edges = cv2.Canny(thresh, 50, 150)

    # Detect axes using Hough Transform
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=100, minLineLength=50, maxLineGap=5)
    if lines is None:
        print(f"No lines detected in {image_path}. Skipping...")
        return None

    width, height = img.shape[1], img.shape[0]
    x_axis_y, y_axis_x = None, None

    for line in lines:
        x1, y1, x2, y2 = line[0]
        if abs(y1 - y2) < 10:
            x_axis_y = y1
        if abs(x1 - x2) < 10:
            y_axis_x = x1

    if x_axis_y is None or y_axis_x is None:
        print(f"Could not detect axes in {image_path}. Skipping...")
        return None

    # Compute scale factors
    x_scale = (graph_x_range[1] - graph_x_range[0]) / (width - y_axis_x)
    y_scale = (graph_y_range[1] - graph_y_range[0]) / (x_axis_y)

    # Find contours of the plotted graph
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    data_points = []
    image_name = os.path.splitext(os.path.basename(image_path))[0]  # Remove extension
    function_class = classify_function(image_name)

    for cnt in contours:
        for point in cnt:
            px, py = point[0]
            real_x = graph_x_range[0] + ((px - y_axis_x) * x_scale)
            real_y = graph_y_range[1] - ((py - x_axis_y) * y_scale)
            distance_from_origin = math.sqrt(real_x**2 + real_y**2)

            data_points.append((real_x, real_y, distance_from_origin, function_class))

    # Convert to DataFrame and process data
    df = pd.DataFrame(data_points, columns=["X", "Y", "Distance_from_Origin", "Class"]).drop_duplicates().sort_values(by=["X"])
    df['Slope'] = df['Y'].diff() / df['X'].diff()
    df['Curvature'] = df['Slope'].diff() / df['X'].diff()
    df['Angle_with_X_Axis'] = np.arctan(df['Slope']) * (180 / np.pi)
    df = df.fillna(0)

    return df

def process_images_from_folder(folder_path, output_csv, graph_x_range=(0, 10), graph_y_range=(-1, 1)):
    """Processes all images in a folder and saves extracted data to a CSV."""
    all_data = pd.DataFrame()

    for file in os.listdir(folder_path):
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_path = os.path.join(folder_path, file)
            df = extract_graph_data(image_path, graph_x_range, graph_y_range)
            if df is not None:
                all_data = pd.concat([all_data, df], ignore_index=True)

    if not all_data.empty:
        all_data.to_csv(output_csv, index=False)
        print(f"Saved extracted data from all images to {output_csv}")

# Example Usage
folder_path = "graphs"  # Change to your folder path
output_csv = "combined_graph_data.csv"
process_images_from_folder(folder_path, output_csv, graph_x_range=(0, 10), graph_y_range=(-10, 10))


Saved extracted data from all images to combined_graph_data.csv
